Use the NHL-API-PY module to scrape master_df from the public NHL API. We're scraping 

In [1]:
# Installs
#!pip install nhl-api-py
#!pip install pandas

# Imports
import pandas as pd
from nhlpy import NHLClient
from nhlpy.api.query.filters.season import SeasonQuery
from nhlpy.api.query.filters.game_type import GameTypeQuery
from nhlpy.api.query.builder import QueryBuilder
import time

# Initialize the client
client = NHLClient()

seasons = ["20232024", "20242025", "20252026"]
reports_to_fetch = ['summary', 'realtime', 'powerplay', 'penaltykill', 'goalsForAgainst', 'percentages']

all_data = []
query_builder = QueryBuilder()

print("Building Master Dataset for Clustering...")

for season in seasons:
    print(f"\nFetching data for {season}...")
    season_df = None
    
    # 1. Build Query Context
    filters = [
        GameTypeQuery(game_type="2"),
        SeasonQuery(season_start=season, season_end=season) 
    ]
    query_context = query_builder.build(filters=filters)
    
    for report in reports_to_fetch:
        print(f"  -> Scraping {report} report...", end=" ")
        
        report_data = []
        start = 0
        limit = 100 # Fetch 100 players at a time
        
        # 2. The Pagination Loop
        try:
            while True:
                response = client.stats.skater_stats_with_query_context(
                    report_type=report,
                    query_context=query_context,
                    aggregate=False, # False ensures distinct individual rows
                    start=start,
                    limit=limit
                )
                
                # Extract the batch data and the total available records
                total = response.get('total', 0) 
                batch_data = response.get('data', [])
                
                report_data.extend(batch_data)
                
                # Break if we've collected everything or the API returns an empty batch
                if len(report_data) >= total or len(batch_data) == 0:
                    break
                    
                start += limit
                time.sleep(0.2) # Small pause between pages
                
            # Convert the fully scraped list for this report into a DataFrame
            df = pd.DataFrame(report_data)
            
            if df.empty:
                print("Warning: No data returned.")
                continue

            # 3. Align and Merge using the strategy we built earlier
            df = df.set_index('playerId')
            
            if season_df is None:
                season_df = df
            else:
                season_df = season_df.combine_first(df)
                
            print(f"Done! ({len(report_data)} players)")
            time.sleep(0.5) # Pause before hitting the next report endpoint
            
        except Exception as e:
            print(f"FAILED: {e}")
            
    # 4. Finalize the season's data
    if season_df is not None and not season_df.empty:
        season_df = season_df.reset_index()
        season_df['seasonId'] = season
        all_data.append(season_df)

# 5. Combine and Clean
print("\nCombining and cleaning data...")
master_df = pd.concat(all_data, ignore_index=True)

# Fill remaining NaNs (like missing PP time) with 0
master_df = master_df.fillna(0)

print("\nDataset Complete! Ready for feature engineering.")
print(f"Total Rows (Player-Seasons): {len(master_df)}")
print(f"Total Columns (Features): {len(master_df.columns)}")


Building Master Dataset for Clustering...

Fetching data for 20232024...
  -> Scraping summary report... Done! (924 players)
  -> Scraping realtime report... Done! (924 players)
  -> Scraping powerplay report... Done! (924 players)
  -> Scraping penaltykill report... Done! (924 players)
  -> Scraping goalsForAgainst report... Done! (924 players)
  -> Scraping percentages report... Done! (924 players)

Fetching data for 20242025...
  -> Scraping summary report... Done! (920 players)
  -> Scraping realtime report... Done! (920 players)
  -> Scraping powerplay report... Done! (920 players)
  -> Scraping penaltykill report... Done! (920 players)
  -> Scraping goalsForAgainst report... Done! (920 players)
  -> Scraping percentages report... Done! (920 players)

Fetching data for 20252026...
  -> Scraping summary report... Done! (910 players)
  -> Scraping realtime report... Done! (910 players)
  -> Scraping powerplay report... Done! (910 players)
  -> Scraping penaltykill report... Done! (9

In [2]:
display(master_df.head())
display(master_df.select_dtypes(include=['number']).info(verbose=True, show_counts=True))

,playerId,assists,blockedShots,blockedShotsPer60,emptyNetAssists,emptyNetGoals,emptyNetPoints,evGoals,evPoints,evenStrengthGoalDifference,...,timeOnIcePerGame,timeOnIcePerGame5v5,totalShotAttempts,usatPercentage,usatPercentageAhead,usatPercentageBehind,usatPercentageTied,usatPrecentageClose,usatRelative,zoneStartPct5v5
0,8470600,15,111,4.28,0.0,0,0.0,2,16,9.0,...,1136.3902,1028.243,254,0.512,0.493,0.525,0.525,0.513,-0.027,0.490
1,8470604,4,33,2.18,1.0,0,1.0,5,9,-5.0,...,754.2361,553.666,175,0.441,0.424,0.492,0.405,0.406,-0.082,0.149
2,8470610,5,25,3.85,0.0,0,0.0,5,8,-7.0,...,778.0333,683.766,78,0.492,0.580,0.422,0.450,0.491,-0.038,0.426
3,8470613,33,87,2.96,1.0,1,2.0,5,22,18.0,...,1290.0487,935.524,536,0.586,0.555,0.618,0.595,0.598,0.001,0.522
4,8470621,10,17,1.41,1.0,0,1.0,9,18,-3.0,...,802.9259,666.962,143,0.489,0.486,0.471,0.513,0.488,-0.027,0.526


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2754 entries, 0 to 2753
Data columns (total 101 columns):
 #    Column                        Non-Null Count  Dtype  
---   ------                        --------------  -----  
 0    playerId                      2754 non-null   int64  
 1    assists                       2754 non-null   int64  
 2    blockedShots                  2754 non-null   int64  
 3    blockedShotsPer60             2754 non-null   float64
 4    emptyNetAssists               2754 non-null   float64
 5    emptyNetGoals                 2754 non-null   int64  
 6    emptyNetPoints                2754 non-null   float64
 7    evGoals                       2754 non-null   int64  
 8    evPoints                      2754 non-null   int64  
 9    evenStrengthGoalDifference    2754 non-null   float64
 10   evenStrengthGoalsAgainst      2754 non-null   float64
 11   evenStrengthGoalsFor          2754 non-null   float64
 12   evenStrengthGoalsForPct       2754 non-null   

None

Now add in more advanced player puck tracking master_df for each player season instance

In [3]:
import pandas as pd
import time
import concurrent.futures
import ast

# 1. Safely convert stringified text back into lists (or pass them through if they already are lists)
def safe_parse(val):
    if isinstance(val, list):
        return val
    if pd.isna(val):
        return []
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except:
            return []
    return []

# 2. Prepare a list of tasks (tuples of playerId and seasonId)
fetch_tasks = [(row['playerId'], row['seasonId']) for index, row in master_df.iterrows()]

edge_master_df_list = []
failed_fetches = []

# 3. Define the worker function
def fetch_edge_data(task):
    p_id, p_season = task
    try:
        edge_stats = client.edge.skater_detail(player_id=p_id, season=p_season)
        
        edge_stats.pop('player', None)
        edge_stats.pop('seasonsWithEdgeStats', None)
        
        edge_stats['playerId'] = p_id
        edge_stats['seasonId'] = p_season
        
        time.sleep(0.05) 
        
        return edge_stats
    except Exception as e:
        return {'error': True, 'playerId': p_id, 'seasonId': p_season, 'msg': str(e)}

print(f"Starting multithreaded fetch for {len(fetch_tasks)} player-seasons...")
start_time = time.time()

# 4. Execute the Thread Pool
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    results = executor.map(fetch_edge_data, fetch_tasks)
    
    for count, result in enumerate(results, 1):
        if 'error' in result:
            failed_fetches.append(result)
        else:
            edge_master_df_list.append(result)
            
        if count % 500 == 0:
            print(f"Processed {count}/{len(fetch_tasks)} records...")

end_time = time.time()
print(f"Fetch complete in {round(end_time - start_time, 2)} seconds!")
print(f"Success: {len(edge_master_df_list)} | Failed: {len(failed_fetches)}")

if failed_fetches:
    print("First 3 failures:", failed_fetches[:3])

# 5. Flatten and Merge EDGE data
print("Flattening and merging EDGE data...")
edge_master_df = pd.json_normalize(edge_master_df_list)

final_df = pd.merge(master_df, edge_master_df, on=['playerId', 'seasonId'], how='left')

cols_to_drop = [c for c in final_df.columns if 'overlay' in str(c)]
final_df = final_df.drop(columns=cols_to_drop)

# ---------------------------------------------------------
# 6. EXTRACT NESTED SHOT DATA
# ---------------------------------------------------------
print("Extracting nested shot location metrics...")

# Ensure columns exist and apply the parser
if 'sogSummary' in final_df.columns and 'sogDetails' in final_df.columns:
    final_df['sogSummary'] = final_df['sogSummary'].apply(safe_parse)
    final_df['sogDetails'] = final_df['sogDetails'].apply(safe_parse)

    # Extract Danger Zones
    final_df['highDangerShots'] = final_df['sogSummary'].apply(
        lambda x: next((item.get('shots', 0) for item in x if item.get('locationCode') == 'high'), 0)
    )
    final_df['highDangerGoals'] = final_df['sogSummary'].apply(
        lambda x: next((item.get('goals', 0) for item in x if item.get('locationCode') == 'high'), 0)
    )
    final_df['midDangerShots'] = final_df['sogSummary'].apply(
        lambda x: next((item.get('shots', 0) for item in x if item.get('locationCode') == 'mid'), 0)
    )
    final_df['longDangerShots'] = final_df['sogSummary'].apply(
        lambda x: next((item.get('shots', 0) for item in x if item.get('locationCode') == 'long'), 0)
    )

    # Extract Specific Ice Areas
    final_df['shotsCrease'] = final_df['sogDetails'].apply(
        lambda x: next((item.get('shots', 0) for item in x if item.get('area') == 'Crease'), 0)
    )
    final_df['shotsLowSlot'] = final_df['sogDetails'].apply(
        lambda x: next((item.get('shots', 0) for item in x if item.get('area') == 'Low Slot'), 0)
    )
    final_df['shotsHighSlot'] = final_df['sogDetails'].apply(
        lambda x: next((item.get('shots', 0) for item in x if item.get('area') == 'High Slot'), 0)
    )

    # Drop the raw dictionary lists
    final_df = final_df.drop(columns=['sogSummary', 'sogDetails'])

print("Pipeline Complete!")

# Output the results
columns_to_preview = ['playerId', 'seasonId', 'highDangerShots', 'shotsCrease', 'shotsLowSlot']
# Find available columns that match our preview list to avoid KeyError
preview_cols = [col for col in columns_to_preview if col in final_df.columns]
display(final_df[preview_cols].head())

print("\nFinal DataFrame Info:")
final_df.info(verbose=True, show_counts=True)

Starting multithreaded fetch for 2754 player-seasons...
Processed 500/2754 records...
Processed 1000/2754 records...
Processed 1500/2754 records...
Processed 2000/2754 records...
Processed 2500/2754 records...
Fetch complete in 148.26 seconds!
Success: 2752 | Failed: 2
First 3 failures: [{'error': True, 'playerId': 8478400, 'seasonId': '20242025', 'msg': 'Request to edge/skater-detail/8478400/20242025/2 failed'}, {'error': True, 'playerId': 8481848, 'seasonId': '20242025', 'msg': 'Request to edge/skater-detail/8481848/20242025/2 failed'}]
Flattening and merging EDGE data...
Extracting nested shot location metrics...
Pipeline Complete!


,playerId,seasonId,highDangerShots,shotsCrease,shotsLowSlot
0,8470600,20232024,1,0,1
1,8470604,20232024,37,4,33
2,8470610,20232024,24,5,19
3,8470613,20232024,3,0,3
4,8470621,20232024,42,5,37



Final DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2754 entries, 0 to 2753
Data columns (total 149 columns):
 #    Column                                     Non-Null Count  Dtype  
---   ------                                     --------------  -----  
 0    playerId                                   2754 non-null   int64  
 1    assists                                    2754 non-null   int64  
 2    blockedShots                               2754 non-null   int64  
 3    blockedShotsPer60                          2754 non-null   float64
 4    emptyNetAssists                            2754 non-null   float64
 5    emptyNetGoals                              2754 non-null   int64  
 6    emptyNetPoints                             2754 non-null   float64
 7    evGoals                                    2754 non-null   int64  
 8    evPoints                                   2754 non-null   int64  
 9    evenStrengthGoalDifference                 2754 non-null   f

In [ ]:
# Clean data by removing nulls

print(f"Starting rows: {len(final_df)}")
cleaned_df = final_df.dropna()
print(f"Rows after blanket dropna: {len(cleaned_df)}")

import numpy as np

# 1. Filter out small sample sizes (Must have played at least 10 games)
cleaned_df = cleaned_df[cleaned_df['gamesPlayed'] >= 10].copy()

print(f"Rows after 10-game filter: {len(cleaned_df)}")

# Convert stats to rate stats if necessary (Per 60, Per 2, Per 5)

# 1. Create time denominators (We use replace(0, np.nan) to prevent dividing by zero)
# Note: timeOnIce in the NHL API is typically in seconds per game.
total_60_blocks = ((cleaned_df['timeOnIcePerGame'] * cleaned_df['gamesPlayed']) / 3600).replace(0, np.nan)
ev_60_blocks = ((cleaned_df['evenStrengthTimeOnIcePerGame'] * cleaned_df['gamesPlayed']) / 3600).replace(0, np.nan)
pp_2_blocks = ((cleaned_df['powerPlayTimeOnIcePerGame'] * cleaned_df['gamesPlayed']) / 120).replace(0, np.nan)
sh_2_blocks = ((cleaned_df['shortHandedTimeOnIcePerGame'] * cleaned_df['gamesPlayed']) / 120).replace(0, np.nan)

# 2. Group raw counting stats into lists based on when they occurred
overall_stats = [
    'goals', 'assists', 'points', 'shots', 'blockedShots', 
    'hits', 'takeaways', 'giveaways', 'missedShots', 'totalShotAttempts'
]

even_strength_stats = ['evGoals', 'evPoints'] 

power_play_stats = [
    'ppGoals', 'ppAssists', 'ppPoints', 'ppShots', 
    'ppPrimaryAssists', 'ppSecondaryAssists', 'ppIndividualSatFor'
]

short_handed_stats = [
    'shGoals', 'shAssists', 'shPoints', 'shShots', 
    'shPrimaryAssists', 'shSecondaryAssists', 'shIndividualSatFor'
]

# 3. Automate the math!
print("Converting stats to rates...")

# Overall -> Per 60
for stat in overall_stats:
    # Check if the column exists to avoid key errors
    if stat in cleaned_df.columns:
        cleaned_df[f'{stat}Per60'] = cleaned_df[stat] / total_60_blocks

# Even Strength -> Per 60
for stat in even_strength_stats:
    if stat in cleaned_df.columns:
        cleaned_df[f'{stat}Per60'] = cleaned_df[stat] / ev_60_blocks

# Power Play -> Per 2
for stat in power_play_stats:
    if stat in cleaned_df.columns:
        cleaned_df[f'{stat}Per2'] = cleaned_df[stat] / pp_2_blocks

# Penalty Kill (Short Handed) -> Per 2
for stat in short_handed_stats:
    if stat in cleaned_df.columns:
        cleaned_df[f'{stat}Per2'] = cleaned_df[stat] / sh_2_blocks

# 4. Clean up the NaNs generated by players who had 0 ice time in a specific category
cleaned_df = cleaned_df.fillna(0)

# Check the results
print("Conversion complete!")
columns_to_preview = ['playerId', 'lastName', 'gamesPlayed', 'hitsPer60', 'blockedShotsPer60', 'ppGoalsPer2', 'shShotsPer2']
display(cleaned_df[columns_to_preview].head(10))

Starting rows: 2754
Rows after blanket dropna: 2732
Rows after 10-game filter: 2299
Converting stats to rates...
Conversion complete!


,playerId,lastName,gamesPlayed,hitsPer60,blockedShotsPer60,ppGoalsPer2,shShotsPer2
0,8470600,Suter,82,3.322459,4.288290,0.000000,0.037736
1,8470604,Carter,72,5.502256,2.187644,0.066409,0.095939
2,8470610,Parise,30,3.084701,3.855876,0.000000,0.000000
3,8470613,Burns,82,1.395296,2.960750,0.037112,0.063571
4,8470621,Perry,54,3.570275,1.411504,0.059573,0.000000
5,8470794,Pavelski,82,3.264734,3.047085,0.100725,0.000000
6,8470966,Giordano,46,4.003227,6.750540,0.000000,0.115367
7,8471214,Ovechkin,79,6.401194,1.066866,0.072296,0.000000
8,8471215,Malkin,82,1.141782,1.653616,0.042806,0.000000
9,8471218,Wheeler,54,2.708409,2.708409,0.038084,0.000000


In [5]:
# Save to a csv

cleaned_df.to_csv('final_player_stats.csv', index=False)